## Python `logging` Module — Fundamentals

### Why not `print()`?

`print()` is fine for a one-off script. In production code you need:

- **Level control** — silence DEBUG noise without deleting code; **change one setting**
- **Automatic context** — timestamp, module name, function name, line number on every message
- **Multiple destinations** — DEBUG to console, INFO+ to a file, ERROR+ to PagerDuty — simultaneously
- **Hierarchy** — configure logging once at app startup; every module in the project inherits the setup

The `logging` module gives you all of this. `print()` gives you none of it.

### The flow of a log record

```shell
your code
  logger.info("order placed")              ← you call this
        │
        ▼
  Logger ("myapp.orders")
    - check: INFO >= effective level? yes → create LogRecord
        │
        ├──► Handler 1: StreamHandler      ← formats + writes to stdout
        ├──► Handler 2: RotatingFileHandler ← formats + writes to app.log
        └──► (propagate=True, default)
               │
               ▼
         parent Logger ("myapp")
               │
               ▼
         root Logger
               └──► any handlers on root
```

Each handler has its **own level filter** and **own formatter** — the same record can appear
differently (or not at all) on the console vs the log file.


## Log levels

Five built-in levels, ordered by severity:

| Level | Numeric value | When to use |
|---|---|---|
| `DEBUG` | 10 | Internal state, variable values, algorithm steps — dev and diagnosis only |
| `INFO` | 20 | Normal operations: requests received, jobs completed, key milestones |
| `WARNING` | 30 | Unexpected but recoverable: deprecated API used, optional config missing |
| `ERROR` | 40 | A failure: exception caught, a task could not complete |
| `CRITICAL` | 50 | System-threatening: database unreachable, out of disk space |


> **Decision rule:** would a sleeping ops engineer want to be paged for this?

- Yes, immediately → `CRITICAL`
- Yes, in the morning → `ERROR`
- Worth noting in daily log review → `WARNING` or `INFO`
- Only useful while actively debugging → `DEBUG`

**`NOTSET` (0):** a logger with level `NOTSET` delegates the level check to its parent.
**Used for child loggers** that inherit the root app level — the typical pattern.
Set the level once on the root app logger; leave all module-level loggers at `NOTSET`.


In [1]:
import logging

# basicConfig() — the simplest possible setup.
# Without force=True, subsequent calls are silently ignored if root already has handlers.
# force=True (Python 3.8+) removes existing root handlers first — safe to re-run in Jupyter.
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

logger = logging.getLogger("demo")

logger.debug("starting up — DEBUG visible because level=DEBUG")
logger.info("user logged in — user_id=42")
logger.warning("rate limit at 85% for ip=1.2.3.4")
logger.error("payment service unreachable — order_id=101")
logger.critical("database connection pool exhausted — all 20 connections in use")


10:04:51 | DEBUG    | demo | starting up — DEBUG visible because level=DEBUG
10:04:51 | INFO     | demo | user logged in — user_id=42
10:04:51 | WARNING  | demo | rate limit at 85% for ip=1.2.3.4
10:04:51 | ERROR    | demo | payment service unreachable — order_id=101
10:04:51 | CRITICAL | demo | database connection pool exhausted — all 20 connections in use


## The singleton pattern

`logging.getLogger(name)` maintains a global dict inside `logging.Manager`.
Every call with the **same name** returns the **exact same object**:

```python
a = logging.getLogger("myapp")
b = logging.getLogger("myapp")
a is b  # True — same object, same id()
```

This is why the canonical pattern works: every module calls `getLogger(__name__)` at import
time and gets a reference to the same object that was configured at app startup.

### The canonical production pattern — one line per module

```python
# At the top of every .py file in your project, after imports:
import logging
logger = logging.getLogger(__name__)
```

`__name__` is the module's fully-qualified dotted path:

| File | `__name__` value |
|---|---|
| `myapp/main.py` | `"myapp.main"` |
| `myapp/orders/views.py` | `"myapp.orders.views"` |
| `myapp/orders/__init__.py` | `"myapp.orders"` |
| entry-point script | `"__main__"` |

This creates a **logger tree that mirrors your package structure:**

```shell
root
 └── myapp
       ├── myapp.main
       └── myapp.orders
             └── myapp.orders.views
```

Configure `logging.getLogger("myapp")` once at startup — every descendant inherits the level
and routes records through it to its handlers (via propagation).


In [4]:
import logging

a = logging.getLogger("myapp")
b = logging.getLogger("myapp")
c = logging.getLogger("myapp.orders")
d = logging.getLogger("myapp.orders.views")

print(f"a is b (same name):                  {a is b}")        # True
print(f"a is c (different name):             {a is c}")        # False
print()
print(f"c.name:                              {c.name!r}")
print(f"c.parent.name:                       {c.parent.name!r}")    # 'myapp'
print(f"d.parent.name:                       {d.parent.name!r}")    # 'myapp.orders'
print()
root = logging.getLogger()
print(f"root logger:                         {root!r}")
print(f"root is parent of 'myapp':           {logging.getLogger('myapp').parent is root}")
print(f"root level (default, no config):     {logging.getLevelName(root.level)}")


a is b (same name):                  True
a is c (different name):             False

c.name:                              'myapp.orders'
c.parent.name:                       'myapp'
d.parent.name:                       'myapp.orders'

root logger:                         <RootLogger root (DEBUG)>
root is parent of 'myapp':           True
root level (default, no config):     DEBUG


### Handlers and Formatters

> A **Handler** decides *where* the record goes.
A **Formatter** decides *how* it is rendered before it gets there.

Each handler has its own level (a second filter after the logger level)
and its own formatter — console and file can show different amounts of detail.

#### Key built-in handlers

| Handler | Destination |
|---|---|
| `logging.StreamHandler` | Any stream — default `sys.stderr`; pass `sys.stdout` explicitly |
| `logging.FileHandler` | A single file (grows without limit) |
| `logging.handlers.RotatingFileHandler` | Rolls over when the file hits `maxBytes` |
| `logging.handlers.TimedRotatingFileHandler` | Rolls over by time: `when="midnight"`, `"h"`, `"W0"` |
| `logging.handlers.SysLogHandler` | OS syslog (Linux / macOS) |
| `logging.NullHandler` | Discards everything — the correct choice for library code |

#### LogRecord format attributes

| Attribute | Format string | Example |
|---|---|---|
| Timestamp | `%(asctime)s` | `2026-07-08 14:30:01` |
| Logger name | `%(name)s` | `myapp.orders` |
| Level string | `%(levelname)s` | `WARNING` |
| Level integer | `%(levelno)d` | `30` |
| The message | `%(message)s` | `Order placed` |
| Module (no .py) | `%(module)s` | `views` |
| Function name | `%(funcName)s` | `create_order` |
| Line number | `%(lineno)d` | `42` |
| Full file path | `%(pathname)s` | `/app/myapp/orders/views.py` |
| Process ID | `%(process)d` | `12345` |
| Thread ID | `%(thread)d` | `140234` |


In [5]:
import logging

# Clear any handlers left over from earlier cells
root = logging.getLogger()
for h in root.handlers[:]:
    root.removeHandler(h)

# ── Logger ─────────────────────────────────────────────────────────────────────
logger = logging.getLogger("myapp")
logger.setLevel(logging.DEBUG)

# Guard: don't add duplicate handlers if this cell is re-run
if not logger.handlers:

    # ── Formatter ──────────────────────────────────────────────────────────────
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)-20s | %(funcName)s:%(lineno)d | %(message)s",
        datefmt="%H:%M:%S",
    )

    # ── Handler ────────────────────────────────────────────────────────────────
    console = logging.StreamHandler()
    console.setLevel(logging.DEBUG)    # second gate: handler level
    console.setFormatter(formatter)

    logger.addHandler(console)

# Child loggers — no handlers, no level; inherit everything via propagation
orders   = logging.getLogger("myapp.orders")
payments = logging.getLogger("myapp.payments")

logger.info("myapp logger configured — both DEBUG and INFO visible")
orders.debug("from myapp.orders — no handler needed on the child")
orders.info("child propagates up to myapp → through myapp's console handler")
payments.warning("myapp.payments also routes through the same myapp handler")


10:58:15 | INFO     | myapp                | <module>:32 | myapp logger configured — both DEBUG and INFO visible
10:58:15 | DEBUG    | myapp.orders         | <module>:33 | from myapp.orders — no handler needed on the child
10:58:15 | INFO     | myapp.orders         | <module>:34 | child propagates up to myapp → through myapp's console handler
10:58:15 | WARNING  | myapp.payments       | <module>:35 | myapp.payments also routes through the same myapp handler


## Propagation — and the double-logging trap

When a logger emits a record:
1. It passes the record to **all of its own handlers**
2. If `propagate=True` (the **default**), it passes the record **up to its parent logger**, which repeats step 1, all the way to root

This is what makes the `__name__` pattern work: child loggers have no handlers
and rely entirely on propagation to reach the configured app logger.

### The trap: double (or triple) logging

If you add a handler to `myapp` AND the root logger also has a handler
(e.g. from `basicConfig()`), every record from `myapp.orders` appears **twice**:

```shell
root  ←── has StreamHandler (from basicConfig)
  ↑
myapp ←── you added ANOTHER StreamHandler here
  ↑
myapp.orders emits logger.info("order placed")
  → passes through myapp's handler   (line 1 ✗)
  → propagates to root's handler     (line 2 ✗)
```

### Two clean fixes

**Fix A — `propagate = False`** — stop the record at `myapp`; it never reaches root.
Use when `myapp` should fully own its output destinations.

**Fix B — remove the handler from `myapp`** — let propagation route to root.
Use when a single root handler covers all your needs.


In [4]:
import logging

# ── Start clean ───────────────────────────────────────────────────────────────
for name in ["", "myapp", "myapp.orders"]:
    lg = logging.getLogger(name)
    for h in lg.handlers[:]:
        lg.removeHandler(h)
    lg.propagate = True
    lg.setLevel(logging.NOTSET)

# ── Simulate: root has a handler (as if basicConfig ran) ──────────────────────
root = logging.getLogger()
root.setLevel(logging.DEBUG)
root.addHandler(logging.StreamHandler())
root.handlers[0].setFormatter(logging.Formatter("[ROOT]  %(name)s | %(message)s"))

# ── Bug: also add a handler directly to myapp ─────────────────────────────────
myapp = logging.getLogger("myapp")
extra = logging.StreamHandler()
extra.setFormatter(logging.Formatter("[MYAPP] %(name)s | %(message)s"))
myapp.addHandler(extra)

orders = logging.getLogger("myapp.orders")

print("=== BUG — double logging (propagate=True + handler on both myapp and root) ===")
orders.warning("payment delayed")

# ── Fix A: propagate=False ────────────────────────────────────────────────────
print()
myapp.propagate = False
print("=== FIX A — propagate=False: record stops at myapp, never reaches root ===")
orders.warning("payment delayed")

# ── Fix B: remove handler from myapp, let propagation do the routing ──────────
print()
myapp.propagate = True
myapp.removeHandler(extra)
print("=== FIX B — no handler on myapp, propagation routes to root ===")
orders.warning("payment delayed")


[MYAPP] myapp.orders | payment delayed


[ROOT]  myapp.orders | payment delayed


[MYAPP] myapp.orders | payment delayed


[ROOT]  myapp.orders | payment delayed


=== BUG — double logging (propagate=True + handler on both myapp and root) ===

=== FIX A — propagate=False: record stops at myapp, never reaches root ===

=== FIX B — no handler on myapp, propagation routes to root ===
